# L2-ARCTIC MDD (Kaggle GPU)

1. Add Data: `tnguynthnh142/l2-arctic1` (`ABA/ABA/wav/` layout)
2. GPU + Internet ON → Run All → download `l2_mdd_best.pt`
_OOM: `batch_size=1`, `grad_accum_steps=8`_


In [ ]:
!pip install -q transformers torchaudio torch-geometric peft tqdm textgrid


In [ ]:
CONFIG = {
  "wavlm": {"model_name": "microsoft/wavlm-large", "freeze": True, "use_lora": False},
  "transformer": {"num_layers": 3, "num_heads": 8, "ff_dim": 2048, "dropout": 0.1, "max_seq_len": 800},
  "ctc_align": {"use_wavlm_ctc_head": True},
  "phoneme_graph": {"hidden_dim": 256, "num_gat_layers": 2, "num_heads": 4, "dropout": 0.1,
    "edge_types": {"sequential": True, "same_word": True, "same_syllable": False}},
  "mdd": {"num_classes": 4, "dropout": 0.1, "class_weights": [1.0, 2.0, 3.0, 3.0]},
  "paths": {"checkpoint_dir": "/kaggle/working/checkpoints/l2_mdd"},
  "train": {"batch_size": 2, "grad_accum_steps": 4, "num_epochs": 40, "learning_rate": 1e-4, "weight_decay": 0.01,
    "grad_clip": 1.0, "num_workers": 0, "pin_memory": True, "use_amp": True, "ctc_weight": 0.1, "save_every_epochs": 5,
    "dataset": {"data_dir": None, "max_duration_sec": 8.0, "sample_rate": 16000}},
}

from pathlib import Path
from typing import List, Tuple

KAGGLE_ROOT = Path("/kaggle/input/datasets/tnguynthnh142/l2-arctic1")
PROBE = ("ABA", "TLV", "NJS")

def speaker_dir(root, spk):
    r = Path(root)
    n = r / spk / spk
    return n if (n / "wav").is_dir() else r / spk

def _ok(root, spk):
    d = speaker_dir(root, spk)
    return (d / "annotation").is_dir() and (d / "wav").is_dir() and any((d / "annotation").glob("*.TextGrid"))

def prepare_l2_arctic_dir(input_root=None):
    seen, cands = set(), [KAGGLE_ROOT, Path("/kaggle/input/l2-arctic1")]
    base = Path(input_root or "/kaggle/input")
    if base.is_dir():
        for tg in base.rglob("annotation/*.TextGrid"):
            inner = tg.parent.parent
            root = inner.parent.parent if inner.parent.name == inner.name else inner.parent
            cands.append(root)
    for c in cands:
        c = Path(c).resolve()
        if str(c) in seen:
            continue
        seen.add(str(c))
        if any(_ok(c, s) for s in PROBE):
            return c
    raise FileNotFoundError("Add Data: tnguynthnh142/l2-arctic1 (expect .../ABA/ABA/wav/)")

def verify_l2_arctic(root):
    root = Path(root)
    if not any(_ok(root, s) for s in PROBE):
        return False, ["Need .../SPEAKER/SPEAKER/wav/ + annotation/"], {"root": str(root)}
    aba = speaker_dir(root, "ABA")
    return True, [], {"root": str(root), "aba": str(aba), "ann": sum(1 for _ in root.rglob("annotation/*.TextGrid"))}

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

TEST_SPEAKERS = frozenset({"TLV", "NJS", "TNI", "TXHC", "ZHAA", "YKWK"})
ALL_SPEAKERS = [
    "ABA", "SKA", "YBAA", "ZHAA", "BWC", "LXC", "NCC", "TXHC", "ASI", "RRBI",
    "SVBI", "TNI", "HJK", "HKK", "YDCK", "YKWK", "EBVS", "ERMS", "MBMPS", "NJS",
    "HQTV", "PNV", "THV", "TLV",
]

ERR_CORRECT, ERR_SUB, ERR_DEL, ERR_ADD = 0, 1, 2, 3
ERR_NAMES = ("correct", "substitution", "deletion", "addition")

DEFAULT_DATA_DIR = None
SAMPLE_RATE = 16000

def speaker_dir(root: Path, spk: str) -> Path:
    
    root = Path(root)
    nested = root / spk / spk
    if (nested / "annotation").is_dir() or (nested / "wav").is_dir():
        return nested
    return root / spk

def _is_sil(ph: str) -> bool:
    return ph.lower() in {"", "sil", "sp", "spn", "pau"}

def _parse_mark(mark: str) -> Tuple[Optional[str], Optional[str], str]:
    t = re.sub(r"[^a-z,]", "", mark.lower())
    if not t:
        return None, None, ""
    if _is_sil(t):
        return "sil", "sil", ""
    parts = t.split(",")
    if len(parts) == 1:
        ph = "ah" if parts[0] == "ax" else parts[0]
        return ph, ph, ""
    cano = "ah" if parts[0] == "ax" else parts[0]
    perc = "ah" if parts[1] == "ax" else parts[1]
    tag = parts[2] if len(parts) > 2 else ""
    return cano, perc, tag

def _load_phones_tier(tg_path: str):
    
    from textgrid import TextGrid

    try:
        tg = TextGrid(strict=False)

        def _lenient_append(tier):
            if tg.maxTime is not None and tier.maxTime is not None and tier.maxTime > tg.maxTime:
                tg.maxTime = tier.maxTime
            tier.strict = False
            for interval in tier:
                interval.strict = False
            tg.tiers.append(tier)

        tg.append = _lenient_append
        tg.read(tg_path)
        return tg.getFirst("phones")
    except Exception:
        return _phones_tier_fallback(tg_path)

def _phones_tier_fallback(tg_path: str):
    text = Path(tg_path).read_text(encoding="utf-8", errors="replace")
    marks: List[str] = []
    in_phones = False
    for line in text.splitlines():
        s = line.strip()
        if re.match(r'name\s*=\s*"phones"', s, re.I):
            in_phones = True
            continue
        if in_phones and re.match(r'name\s*=\s*"', s, re.I) and "phones" not in s.lower():
            break
        if in_phones and re.search(r"text\s*=", s, re.I):
            m = re.search(r'=\s*"(.*)"\s*$', s)
            if m:
                marks.append(m.group(1))
    if not marks:
        return None

    class _Interval:
        __slots__ = ("mark",)

        def __init__(self, mark: str):
            self.mark = mark

    return [_Interval(m) for m in marks]

def parse_textgrid(tg_path: str) -> Tuple[List[str], List[int]]:
    
    tier = _load_phones_tier(tg_path)
    if tier is None:
        return [], []

    canonical, labels = [], []
    for interval in tier:
        cano, perc, tag = _parse_mark(interval.mark)
        if cano is None or _is_sil(cano):
            continue
        if tag == "a":
            if perc and not _is_sil(perc):
                canonical.append(perc)
                labels.append(ERR_ADD)
            continue
        if tag == "d" or perc is None:
            canonical.append(cano)
            labels.append(ERR_DEL)
        elif tag == "s" or (perc and perc != cano):
            canonical.append(cano)
            labels.append(ERR_SUB)
        else:
            canonical.append(cano)
            labels.append(ERR_CORRECT)

    return canonical, labels

def load_l2_arctic_annotated(
    split: str = "train",
    data_dir: Optional[str] = None,
) -> List[Dict[str, Any]]:
    root = Path(data_dir)
    test = split == "test"
    speakers = [s for s in ALL_SPEAKERS if (s in TEST_SPEAKERS) == test]

    samples = []
    for spk in speakers:
        spk_root = speaker_dir(root, spk)
        ann_dir = spk_root / "annotation"
        if not ann_dir.is_dir():
            continue
        for tg in sorted(ann_dir.glob("*.TextGrid")):
            stem = tg.stem
            wav = spk_root / "wav" / f"{stem}.wav"
            txt = spk_root / "transcript" / f"{stem}.txt"
            if not wav.is_file():
                continue
            phones, labels = parse_textgrid(str(tg))
            if not phones:
                continue
            text = txt.read_text(encoding="utf-8").strip() if txt.is_file() else ""
            word_ranges = _uniform_word_ranges(phones, text)
            samples.append(
                {
                    "id": f"{spk}_{stem}",
                    "speaker": spk,
                    "audio": str(wav),
                    "text": text,
                    "phoneme_tokens": phones,
                    "phoneme_labels": labels,
                    "word_phone_ranges": word_ranges,
                }
            )
    return samples

def _uniform_word_ranges(phones: List[str], text: str) -> List[Tuple[int, int]]:
    words = text.split()
    if not words:
        return [(0, len(phones))]
    n = len(words)
    chunk = max(len(phones) // n, 1)
    ranges = []
    for i in range(n):
        s = i * chunk
        e = len(phones) if i == n - 1 else min((i + 1) * chunk, len(phones))
        ranges.append((s, e))
    return ranges

class L2ArcticMDDDataset(Dataset):
    def __init__(
        self,
        split: str = "train",
        sample_rate: int = SAMPLE_RATE,
        max_duration_sec: float = 15.0,
        data_dir: Optional[str] = None,
    ):
        self.sample_rate = sample_rate
        self.max_samples = int(max_duration_sec * sample_rate)
        self.samples = load_l2_arctic_annotated(split, data_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def _load_audio(self, path: str) -> torch.Tensor:
        import torchaudio

        wav, sr = torchaudio.load(path)
        wav = wav.mean(0)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        if wav.shape[0] > self.max_samples:
            wav = wav[: self.max_samples]
        return wav

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.samples[idx]
        return {
            "id": item["id"],
            "waveform": self._load_audio(item["audio"]),
            "text": item["text"],
            "phoneme_tokens": item["phoneme_tokens"],
            "phoneme_labels": item["phoneme_labels"],
            "word_phone_ranges": item["word_phone_ranges"],
        }

def collate_mdd_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    waveforms = [b["waveform"] for b in batch]
    return {
        "ids": [b["id"] for b in batch],
        "waveforms": pad_sequence(waveforms, batch_first=True),
        "wav_lengths": torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long),
        "texts": [b["text"] for b in batch],
        "phoneme_tokens": [b["phoneme_tokens"] for b in batch],
        "phoneme_labels": [b["phoneme_labels"] for b in batch],
        "word_phone_ranges": [b["word_phone_ranges"] for b in batch],
    }

In [ ]:
from contextlib import nullcontext
from typing import Optional

import torch
import torch.nn as nn
from transformers import WavLMModel

class WavLMEncoder(nn.Module):
    

    def __init__(
        self,
        model_name: str = "microsoft/wavlm-large",
        freeze: bool = True,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.05,
        lora_target_modules: Optional[list] = None,
    ):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.output_dim = self.wavlm.config.hidden_size  # 1024 for large

        if freeze and not use_lora:
            for p in self.wavlm.parameters():
                p.requires_grad = False
            self.wavlm.eval()

        self._frozen = freeze and not use_lora

        if use_lora:
            from peft import LoraConfig, get_peft_model

            target = lora_target_modules or ["q_proj", "v_proj"]
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=target,
                lora_dropout=lora_dropout,
                bias="none",
            )
            self.wavlm = get_peft_model(self.wavlm, lora_config)

    def forward(
        self,
        waveform: torch.Tensor,
        wav_lengths: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        
        if attention_mask is None:
            B, S = waveform.shape
            if wav_lengths is not None:
                attention_mask = (
                    torch.arange(S, device=waveform.device).unsqueeze(0)
                    < wav_lengths.unsqueeze(1)
                ).long()
            else:
                attention_mask = (waveform.abs() > 1e-8).long()

        if self._frozen:
            self.wavlm.eval()

        ctx = torch.inference_mode if self._frozen else nullcontext
        with ctx():
            outputs = self.wavlm(
                input_values=waveform,
                attention_mask=attention_mask,
            )
        return outputs.last_hidden_state

    def frame_lengths_from_samples(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        
        return self.wavlm._get_feat_extract_output_lengths(wav_lengths).long()

    def frame_rate(self, sample_rate: int = 16000) -> float:
        
        return sample_rate / 320.0

import math
from typing import Optional

import torch
import torch.nn as nn

class TaskTransformerEncoder(nn.Module):
    

    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        num_heads: int = 8,
        ff_dim: int = 4096,
        dropout: float = 0.1,
        max_seq_len: int = 2000,
    ):
        super().__init__()
        if input_dim % num_heads != 0:
            raise ValueError(
                f"input_dim ({input_dim}) must be divisible by num_heads ({num_heads})"
            )
        self.input_proj = nn.Linear(input_dim, input_dim)
        self.pos_encoding = SinusoidalPositionalEncoding(input_dim, max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_dim = input_dim

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)

class SinusoidalPositionalEncoding(nn.Module):
    

    def __init__(self, dim: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float) * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

torchaudio = None
forced_align = None

def _ensure_torchaudio():
    global torchaudio, forced_align
    if torchaudio is not None:
        return True
    try:
        import torchaudio as _ta
        from torchaudio.functional import forced_align as _fa

        torchaudio = _ta
        forced_align = _fa
        return True
    except (ImportError, OSError):
        return False

DEFAULT_PHONEME_VOCAB = [
    "<pad>", "<unk>", "|",  # blank, unknown, word boundary
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2",
    "AO0", "AO1", "AO2", "AW0", "AW1", "AW2", "AY0", "AY1", "AY2",
    "B", "CH", "D", "DH", "EH0", "EH1", "EH2", "ER0", "ER1", "ER2",
    "EY0", "EY1", "EY2", "F", "G", "HH", "IH0", "IH1", "IH2",
    "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH",
    "T", "TH", "UH0", "UH1", "UH2", "UW0", "UW1", "UW2",
    "V", "W", "Y", "Z", "ZH",
]

@dataclass
class PhonemeAlignment:
    

    phoneme: str
    token_id: int
    start_frame: int
    end_frame: int
    confidence: float

class CTCAligner(nn.Module):
    

    def __init__(
        self,
        input_dim: int,
        phoneme_vocab: Optional[List[str]] = None,
        use_pretrained_bundle: bool = False,
        blank_id: int = 0,
    ):
        super().__init__()
        self.phoneme_vocab = phoneme_vocab or DEFAULT_PHONEME_VOCAB
        self.token2id = {p: i for i, p in enumerate(self.phoneme_vocab)}
        self.blank_id = blank_id
        self.num_tokens = len(self.phoneme_vocab)

        self.ctc_proj = nn.Linear(input_dim, self.num_tokens)

    def phonemes_to_ids(self, phonemes: List[str]) -> torch.Tensor:
        
        ids = []
        for p in phonemes:
            ids.append(self.token2id.get(p, self.token2id["<unk>"]))
        return torch.tensor(ids, dtype=torch.long)

    def forward_ctc_logits(self, frame_features: torch.Tensor) -> torch.Tensor:
        
        logits = self.ctc_proj(frame_features)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.transpose(0, 1)

    def _align_log_probs(self, frame_features: torch.Tensor) -> torch.Tensor:
        
        logits = self.ctc_proj(frame_features.unsqueeze(0))
        return F.log_softmax(logits, dim=-1)

    def _run_forced_align(
        self,
        log_probs: torch.Tensor,
        targets: torch.Tensor,
        input_lengths: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        
        assert log_probs.dim() == 3 and log_probs.shape[0] == 1
        try:
            return forced_align(
                log_probs.float(),
                targets.unsqueeze(0),
                input_lengths,
                target_lengths,
                blank=self.blank_id,
            )
        except RuntimeError:
            return forced_align(
                log_probs.float().cpu(),
                targets.unsqueeze(0).cpu(),
                input_lengths.cpu(),
                target_lengths.cpu(),
                blank=self.blank_id,
            )

    def align_utterance(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
        frame_lengths: Optional[int] = None,
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        
        if frame_lengths is None:
            frame_lengths = frame_features.shape[0]

        T, D = frame_features.shape
        if len(phonemes) == 0:
            return [], frame_features.new_zeros(0, D)

        targets = self.phonemes_to_ids(phonemes).to(frame_features.device)

        if not _ensure_torchaudio() or forced_align is None:
            return self._uniform_align(frame_features, phonemes)

        log_probs = self._align_log_probs(frame_features)  # (1, T, C)
        input_lengths = torch.tensor([frame_lengths], device=frame_features.device)
        target_lengths = torch.tensor([len(targets)], device=frame_features.device)

        aligned_tokens, _align_scores = self._run_forced_align(
            log_probs, targets, input_lengths, target_lengths
        )
        if aligned_tokens.dim() > 1:
            aligned = aligned_tokens[0, :frame_lengths]
        else:
            aligned = aligned_tokens[:frame_lengths]

        spans = self._tokens_to_spans(aligned, targets, phonemes)
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def _tokens_to_spans(
        self,
        aligned: torch.Tensor,
        targets: torch.Tensor,
        phonemes: List[str],
    ) -> List[PhonemeAlignment]:
        
        spans: List[PhonemeAlignment] = []
        target_list = targets.tolist()
        i = 0
        while i < len(phonemes):
            token_id = target_list[i]
            mask = aligned == i  # forced_align uses target position indices
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0]
                start_f, end_f = int(idx[0]), int(idx[-1])
                conf = float(mask.float().mean())
            else:
                start_f = end_f = 0
                conf = 0.0
            spans.append(
                PhonemeAlignment(
                    phoneme=phonemes[i],
                    token_id=token_id,
                    start_frame=start_f,
                    end_frame=end_f,
                    confidence=conf,
                )
            )
            i += 1
        return spans

    def _pool_node_features(
        self,
        frame_features: torch.Tensor,
        spans: List[PhonemeAlignment],
    ) -> torch.Tensor:
        
        nodes = []
        T = frame_features.shape[0]
        for span in spans:
            s = max(0, span.start_frame)
            e = min(T - 1, span.end_frame)
            if s <= e:
                pooled = frame_features[s : e + 1].mean(dim=0)
            else:
                pooled = frame_features.mean(dim=0)
            nodes.append(pooled)
        return torch.stack(nodes, dim=0) if nodes else frame_features.new_zeros(0, frame_features.shape[-1])

    def _uniform_align(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        
        T, D = frame_features.shape
        n = max(len(phonemes), 1)
        chunk = T // n
        spans = []
        for i, ph in enumerate(phonemes):
            s = i * chunk
            e = min(T - 1, (i + 1) * chunk - 1) if i < n - 1 else T - 1
            tid = self.token2id.get(ph, self.token2id["<unk>"])
            spans.append(
                PhonemeAlignment(ph, tid, s, e, 1.0)
            )
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def batch_align(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> List[Tuple[List[PhonemeAlignment], torch.Tensor]]:
        
        results = []
        B = frame_features.shape[0]
        for b in range(B):
            T_b = int(frame_lengths[b].item())
            spans, nodes = self.align_utterance(
                frame_features[b, :T_b],
                phoneme_lists[b],
                T_b,
            )
            results.append((spans, nodes))
        return results

    def ctc_loss(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> torch.Tensor:
        
        log_probs = self.forward_ctc_logits(frame_features)
        targets_list = []
        target_lengths = []
        for phs in phoneme_lists:
            t = self.phonemes_to_ids(phs)
            targets_list.append(t)
            target_lengths.append(len(t))
        targets = torch.cat(targets_list).to(frame_features.device)
        target_lengths_t = torch.tensor(target_lengths, device=frame_features.device)
        loss = F.ctc_loss(
            log_probs,
            targets,
            frame_lengths,
            target_lengths_t,
            blank=self.blank_id,
            zero_infinity=True,
        )
        return loss

from typing import List, Optional, Tuple

import torch
import torch.nn as nn

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    GATv2Conv = None

class PhonemeGraphNetwork(nn.Module):
    

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        edge_sequential: bool = True,
        edge_same_word: bool = True,
        edge_same_syllable: bool = False,
    ):
        super().__init__()
        if GATv2Conv is None:
            raise ImportError("torch_geometric is required for PhonemeGraphNetwork")

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.edge_sequential = edge_sequential
        self.edge_same_word = edge_same_word
        self.edge_same_syllable = edge_same_syllable

        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            in_ch = hidden_dim
            out_ch = hidden_dim // num_heads
            self.gat_layers.append(
                GATv2Conv(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    heads=num_heads,
                    dropout=dropout,
                    concat=True,
                )
            )
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim

    @staticmethod
    def build_edge_index(
        num_phonemes: int,
        word_phone_ranges: List[Tuple[int, int]],
        sequential: bool = True,
        same_word: bool = True,
        same_syllable: bool = False,
    ) -> torch.Tensor:
        
        edges = set()

        if sequential:
            for i in range(num_phonemes - 1):
                edges.add((i, i + 1))
                edges.add((i + 1, i))

        if same_word:
            for start, end in word_phone_ranges:
                for i in range(start, end):
                    for j in range(start, end):
                        if i != j:
                            edges.add((i, j))

        if same_syllable:
            for start, end in word_phone_ranges:
                mid = (start + end) // 2
                for i in range(start, mid):
                    for j in range(start, mid):
                        if i != j:
                            edges.add((i, j))
                for i in range(mid, end):
                    for j in range(mid, end):
                        if i != j:
                            edges.add((i, j))

        if not edges:
            edges.add((0, 0))

        src, dst = zip(*edges)
        return torch.tensor([src, dst], dtype=torch.long)

    def forward_single(
        self,
        node_features: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        
        x = self.input_proj(node_features)
        edge_index = self.build_edge_index(
            node_features.shape[0],
            word_phone_ranges,
            self.edge_sequential,
            self.edge_same_word,
            self.edge_same_syllable,
        ).to(node_features.device)

        for gat in self.gat_layers:
            x = gat(x, edge_index)
            x = self.dropout(torch.relu(x))
        return self.norm(x)

    def forward_batch(
        self,
        node_features_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[torch.Tensor]:
        
        outputs = []
        for nodes, ranges in zip(node_features_list, word_phone_ranges_list):
            if nodes.shape[0] == 0:
                outputs.append(nodes)
            else:
                outputs.append(self.forward_single(nodes, ranges))
        return outputs

from typing import Dict, List

import torch
import torch.nn as nn
import torch.nn.functional as F

ERR_CORRECT, ERR_SUB, ERR_DEL, ERR_ADD = 0, 1, 2, 3
ERR_NAMES = ("correct", "substitution", "deletion", "addition")

class PhonemeMDDHead(nn.Module):
    

    def __init__(self, input_dim: int, num_classes: int = 4, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, num_classes),
        )

    def forward(self, node_features: torch.Tensor) -> torch.Tensor:
        
        return self.net(node_features)

class MDDLoss(nn.Module):
    

    def __init__(self, class_weights: List[float] | None = None):
        super().__init__()
        w = torch.tensor(class_weights or [1.0, 2.0, 3.0, 3.0])
        self.register_buffer("weight", w)

    def forward(
        self,
        logits_list: List[torch.Tensor],
        label_lists: List[List[int]],
    ) -> Dict[str, torch.Tensor]:
        device = logits_list[0].device if logits_list else self.weight.device
        weight = self.weight.to(device)
        losses, all_pred, all_true = [], [], []
        for logits, labels in zip(logits_list, label_lists):
            n = min(logits.shape[0], len(labels))
            if n == 0:
                continue
            tgt = torch.tensor(labels[:n], device=device, dtype=torch.long)
            losses.append(F.cross_entropy(logits[:n], tgt, weight=weight, ignore_index=-1))
            all_pred.append(logits[:n].argmax(-1))
            all_true.append(tgt)

        if not losses:
            z = torch.tensor(0.0, device=device, requires_grad=True)
            return {"loss": z, "accuracy": z}

        loss = torch.stack(losses).mean()
        pred = torch.cat(all_pred)
        true = torch.cat(all_true)
        acc = (pred == true).float().mean()
        err_pred = pred != ERR_CORRECT
        err_true = true != ERR_CORRECT
        tp = (err_pred & err_true).sum().float()
        fp = (err_pred & ~err_true).sum().float()
        fn = (~err_pred & err_true).sum().float()
        prec = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        f1 = 2 * prec * rec / (prec + rec + 1e-8)
        return {"loss": loss, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn

class L2MDDModel(nn.Module):
    

    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        wavlm_cfg = config.get("wavlm", {})
        trans_cfg = config.get("transformer", {})
        ctc_cfg = config.get("ctc_align", {})
        graph_cfg = config.get("phoneme_graph", {})
        mdd_cfg = config.get("mdd", {})

        self.wavlm = WavLMEncoder(
            model_name=wavlm_cfg.get("model_name", "microsoft/wavlm-large"),
            freeze=wavlm_cfg.get("freeze", True),
            use_lora=wavlm_cfg.get("use_lora", False),
        )
        d = self.wavlm.output_dim

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=trans_cfg.get("num_layers", 3),
            num_heads=trans_cfg.get("num_heads", 8),
            ff_dim=trans_cfg.get("ff_dim", 4096),
            dropout=trans_cfg.get("dropout", 0.1),
            max_seq_len=trans_cfg.get("max_seq_len", 2000),
        )

        self.ctc_aligner = CTCAligner(input_dim=d, use_pretrained_bundle=False)
        graph_hidden = graph_cfg.get("hidden_dim", 256)
        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=graph_cfg.get("num_gat_layers", 2),
            num_heads=graph_cfg.get("num_heads", 4),
            dropout=graph_cfg.get("dropout", 0.1),
            edge_sequential=graph_cfg.get("edge_types", {}).get("sequential", True),
            edge_same_word=graph_cfg.get("edge_types", {}).get("same_word", True),
            edge_same_syllable=graph_cfg.get("edge_types", {}).get("same_syllable", False),
        )
        self.mdd_head = PhonemeMDDHead(
            graph_hidden,
            num_classes=mdd_cfg.get("num_classes", 4),
            dropout=mdd_cfg.get("dropout", 0.1),
        )

    def forward(
        self,
        waveforms: torch.Tensor,
        wav_lengths: torch.Tensor,
        phoneme_tokens: List[List[str]],
        word_phone_ranges: List[List[tuple]],
    ) -> Dict[str, Any]:
        frame_feats = self.wavlm(waveforms, wav_lengths=wav_lengths)
        frame_lengths = self.wavlm.frame_lengths_from_samples(wav_lengths)

        T = frame_feats.shape[1]
        pad_mask = torch.arange(T, device=waveforms.device).unsqueeze(0) >= frame_lengths.unsqueeze(1)
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        align_results = self.ctc_aligner.batch_align(frame_feats, phoneme_tokens, frame_lengths)
        node_features_list = [nodes for _, nodes in align_results]

        graph_out = self.phoneme_graph.forward_batch(node_features_list, word_phone_ranges)
        logits_list = [self.mdd_head(g) for g in graph_out]

        ctc_loss = self.ctc_aligner.ctc_loss(frame_feats, phoneme_tokens, frame_lengths)

        return {
            "mdd_logits": logits_list,
            "ctc_loss": ctc_loss,
            "graph_embeddings": graph_out,
        }

    @classmethod
    def from_checkpoint(cls, path: str, config: Dict[str, Any], device: str = "cpu") -> "L2MDDModel":
        model = cls(config).to(device)
        st = torch.load(path, map_location=device, weights_only=False)
        model.load_state_dict(st["model_state_dict"] if isinstance(st, dict) and "model_state_dict" in st else st)
        model.eval()
        return model

import logging
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

logger = logging.getLogger(__name__)

class MDDTrainer:
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        tc = config["train"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.use_amp = bool(tc.get("use_amp", False)) and self.device.type == "cuda"
        self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
        ds_kw = dict(
            sample_rate=tc["dataset"]["sample_rate"],
            max_duration_sec=tc["dataset"]["max_duration_sec"],
            data_dir=tc["dataset"].get("data_dir"),
        )
        self.train_ds = L2ArcticMDDDataset("train", **ds_kw)
        self.val_ds = L2ArcticMDDDataset("test", **ds_kw)
        if len(self.train_ds) == 0:
            raise FileNotFoundError(
                "No L2-ARCTIC annotated samples found. "
                "Download corpus to data/l2-arctic/ (see https://psi.engr.tamu.edu/l2-arctic-corpus/)"
            )
        pin = bool(tc.get("pin_memory", False)) and self.device.type == "cuda"
        self.train_loader = DataLoader(
            self.train_ds, tc["batch_size"], shuffle=True,
            num_workers=tc.get("num_workers", 2), collate_fn=collate_mdd_fn, pin_memory=pin,
        )
        self.val_loader = DataLoader(
            self.val_ds, tc["batch_size"], shuffle=False,
            num_workers=tc.get("num_workers", 2), collate_fn=collate_mdd_fn, pin_memory=pin,
        )
        self.model = L2MDDModel(config).to(self.device)
        mdd_cfg = config.get("mdd", {})
        self.criterion = MDDLoss(mdd_cfg.get("class_weights")).to(self.device)
        self.ctc_weight = tc.get("ctc_weight", 0.1)
        self.optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=tc["learning_rate"], weight_decay=tc.get("weight_decay", 0.01),
        )
        self.num_epochs = tc["num_epochs"]
        self.grad_clip = tc.get("grad_clip", 1.0)
        self.grad_accum = max(1, int(tc.get("grad_accum_steps", 1)))
        self.save_every = tc.get("save_every_epochs", 5)
        self.ckpt_dir = Path(config["paths"]["checkpoint_dir"])
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

    def _step(self, batch, train=True, accum_scale=1.0):
        waveforms = batch["waveforms"].to(self.device)
        wav_lengths = batch["wav_lengths"].to(self.device)
        ctx = torch.cuda.amp.autocast if train and self.use_amp else nullcontext
        with ctx():
            out = self.model(waveforms, wav_lengths, batch["phoneme_tokens"], batch["word_phone_ranges"])
            ld = self.criterion(out["mdd_logits"], batch["phoneme_labels"])
            loss = (ld["loss"] + self.ctc_weight * out["ctc_loss"]) / accum_scale
        if train:
            if self.use_amp:
                self.scaler.scale(loss).backward()
            else:
                loss.backward()
        ld = dict(ld)
        ld["loss"] = loss.detach() * accum_scale
        return ld

    def _optimizer_step(self):
        if self.use_amp:
            if self.grad_clip:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            if self.grad_clip:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
        self.optimizer.zero_grad()

    def train(self):
        logging.basicConfig(level=logging.INFO)
        logger.info("Device=%s train=%d val=%d", self.device, len(self.train_ds), len(self.val_ds))
        best_f1 = 0.0
        for epoch in range(1, self.num_epochs + 1):
            t0 = time.time()
            self.model.train()
            tr = {k: 0.0 for k in ("loss", "f1", "accuracy")}
            n = 0
            self.optimizer.zero_grad()
            for i, batch in enumerate(tqdm(self.train_loader, desc=f"Epoch {epoch}")):
                ld = self._step(batch, True, self.grad_accum)
                for k in tr:
                    if k in ld:
                        tr[k] += float(ld[k].cpu())
                n += 1
                if i % self.grad_accum == self.grad_accum - 1 or i == len(self.train_loader) - 1:
                    self._optimizer_step()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
            self.model.eval()
            va = {k: 0.0 for k in ("loss", "f1", "accuracy", "precision", "recall")}
            m = 0
            with torch.no_grad():
                for batch in tqdm(self.val_loader, desc="Val"):
                    ld = self._step(batch, False)
                    for k in va:
                        if k in ld:
                            va[k] += float(ld[k].cpu())
                    m += 1
            if self.device.type == "cuda":
                torch.cuda.empty_cache()
            tr = {k: v / max(n, 1) for k, v in tr.items()}
            va = {k: v / max(m, 1) for k, v in va.items()}
            logger.info(
                "Epoch %d (%.0fs) train loss=%.4f f1=%.3f | val loss=%.4f f1=%.3f P=%.3f R=%.3f",
                epoch, time.time() - t0, tr["loss"], tr.get("f1", 0),
                va["loss"], va.get("f1", 0), va.get("precision", 0), va.get("recall", 0),
            )
            if va.get("f1", 0) > best_f1:
                best_f1 = va["f1"]
                torch.save(self.model.state_dict(), self.ckpt_dir / "best_model.pt")
            if epoch % self.save_every == 0:
                torch.save(
                    {"epoch": epoch, "model_state_dict": self.model.state_dict(), "config": self.config},
                    self.ckpt_dir / f"epoch_{epoch}.pt",
                )

In [ ]:
import os, shutil, torch
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
assert torch.cuda.is_available(), "Enable GPU T4"
torch.cuda.empty_cache()

data_dir = prepare_l2_arctic_dir()
CONFIG["train"]["dataset"]["data_dir"] = str(data_dir)
ok, issues, info = verify_l2_arctic(data_dir)
print("Dataset:", info)
if not ok:
    raise RuntimeError("\n".join(issues))
print("Samples:", len(L2ArcticMDDDataset("train", data_dir=str(data_dir))), "train,",
      len(L2ArcticMDDDataset("test", data_dir=str(data_dir))), "test")
MDDTrainer(CONFIG).train()
shutil.copy("/kaggle/working/checkpoints/l2_mdd/best_model.pt", "/kaggle/working/l2_mdd_best.pt")
print("Done → l2_mdd_best.pt")